# 🔍 Detecção de Fraudes em Transações Financeiras

> **Problema de negócio:** Identificar automaticamente transações fraudulentas em um dataset altamente desbalanceado, minimizando perdas financeiras e falsos positivos que afetam a experiência do cliente.

**Dataset:** [Credit Card Fraud Detection — Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Técnicas:** EDA, SMOTE, Isolation Forest, XGBoost, SHAP  
**Métricas foco:** Precision-Recall AUC, F1-Score, KS Statistic

---

## Setup e importações

In [ ]:
# Instalar dependências (rodar uma vez)
# !pip install xgboost imbalanced-learn shap scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)

# Estilo dos gráficos
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'fraud': '#E24B4A', 'legit': '#185FA5', 'neutral': '#888780'}

print('Setup concluído ✓')

## 1. Carregamento e visão inicial dos dados

O dataset contém transações de cartão de crédito de setembro de 2013, com 284.807 transações das quais apenas 492 (0,17%) são fraudes. As features V1–V28 são componentes principais obtidas via PCA por razões de confidencialidade. Apenas `Time` e `Amount` estão em escala original.

In [ ]:
# Carregue o arquivo após baixar do Kaggle
df = pd.read_csv('creditcard.csv')

print(f'Shape: {df.shape}')
print(f'\nFraudes: {df["Class"].sum():,} ({df["Class"].mean()*100:.3f}%)')
print(f'Legítimas: {(df["Class"]==0).sum():,} ({(1-df["Class"].mean())*100:.3f}%)')
df.head()

In [ ]:
# Verificar missing values e tipos
print('Missing values:', df.isnull().sum().sum())
print('\nTipos de dados:')
print(df.dtypes.value_counts())
print('\nEstatísticas básicas — Amount:')
df['Amount'].describe()

## 2. Análise Exploratória (EDA)

### 2.1 O problema do desbalanceamento

Este é o desafio central do projeto. Um modelo que sempre prevê "não fraude" teria 99,83% de acurácia — mas seria completamente inútil. Por isso, usaremos métricas adequadas e técnicas de balanceamento.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribuição das classes
class_counts = df['Class'].value_counts()
axes[0].bar(['Legítima', 'Fraude'], class_counts.values,
            color=[COLORS['legit'], COLORS['fraud']], width=0.4)
axes[0].set_title('Distribuição das classes (escala log)', fontsize=12)
axes[0].set_yscale('log')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v * 1.1, f'{v:,}', ha='center', fontsize=10)

# Distribuição do valor das transações por classe
df[df['Class']==0]['Amount'].hist(bins=50, ax=axes[1], alpha=0.6,
                                   color=COLORS['legit'], label='Legítima')
df[df['Class']==1]['Amount'].hist(bins=50, ax=axes[1], alpha=0.7,
                                   color=COLORS['fraud'], label='Fraude')
axes[1].set_title('Distribuição do valor das transações', fontsize=12)
axes[1].set_xlabel('Valor (€)')
axes[1].legend()
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('imgs/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nInsight: Fraudes tendem a ser de valor menor — possível estratégia para fugir de detecção.')
print(f'Mediana fraude: €{df[df["Class"]==1]["Amount"].median():.2f}')
print(f'Mediana legítima: €{df[df["Class"]==0]["Amount"].median():.2f}')

In [ ]:
# Distribuição temporal das fraudes
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

df[df['Class']==0]['Time'].hist(bins=48, ax=axes[0], color=COLORS['legit'], alpha=0.7)
axes[0].set_title('Transações legítimas ao longo do tempo', fontsize=11)
axes[0].set_ylabel('Contagem')

df[df['Class']==1]['Time'].hist(bins=48, ax=axes[1], color=COLORS['fraud'], alpha=0.7)
axes[1].set_title('Fraudes ao longo do tempo', fontsize=11)
axes[1].set_xlabel('Tempo (segundos desde primeira transação)')
axes[1].set_ylabel('Contagem')

plt.tight_layout()
plt.savefig('imgs/02_time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Quais features PCA têm maior separação entre fraude e legítima?
fraud = df[df['Class']==1]
legit = df[df['Class']==0]

pca_features = [f'V{i}' for i in range(1, 29)]
separations = {}
for col in pca_features:
    sep = abs(fraud[col].mean() - legit[col].mean())
    separations[col] = sep

sep_series = pd.Series(separations).sort_values(ascending=False)

plt.figure(figsize=(12, 4))
colors = [COLORS['fraud'] if v > sep_series.median() else COLORS['neutral']
          for v in sep_series.values]
sep_series.plot(kind='bar', color=colors)
plt.title('Separação entre fraude e legítima por feature (diferença absoluta de médias)', fontsize=12)
plt.xlabel('Feature')
plt.ylabel('|mean_fraude - mean_legítima|')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('imgs/03_feature_separation.png', dpi=150, bbox_inches='tight')
plt.show()

top_features = sep_series.head(5).index.tolist()
print(f'Top 5 features mais discriminativas: {top_features}')

## 3. Pré-processamento

### Por que não usar acurácia?

Com 0,17% de fraudes, um modelo "burro" que nunca detecta fraude teria **99,83% de acurácia**. Usaremos:
- **Precision**: dos que o modelo apontou como fraude, quantos eram reais?
- **Recall**: das fraudes reais, quantas o modelo encontrou?
- **F1**: média harmônica entre precision e recall
- **PR-AUC**: área sob a curva Precision-Recall — mais informativa que ROC em datasets desbalanceados
- **KS Statistic**: muito usado em scorecards de crédito em bancos

In [ ]:
# Criar pasta para imagens
import os
os.makedirs('imgs', exist_ok=True)

# Normalizar Amount e Time (V1-V28 já estão normalizadas via PCA)
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])

# Separar features e target
feature_cols = pca_features + ['Amount_scaled', 'Time_scaled']
X = df[feature_cols]
y = df['Class']

# Split estratificado (mantém proporção de fraudes em treino e teste)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Treino: {X_train.shape[0]:,} amostras | Fraudes: {y_train.sum()} ({y_train.mean()*100:.2f}%)')
print(f'Teste:  {X_test.shape[0]:,} amostras | Fraudes: {y_test.sum()} ({y_test.mean()*100:.2f}%)')

## 4. Modelagem

Vamos comparar 4 abordagens:
1. **Baseline**: Regressão Logística (sem balanceamento)
2. **Regressão Logística + SMOTE** (oversampling sintético)
3. **XGBoost + class_weight** (modelo robusto com peso nas classes)
4. **Isolation Forest** (detecção de anomalias não-supervisionada)

In [ ]:
def evaluate_model(name, y_true, y_pred, y_proba):
    """Calcula e retorna métricas relevantes para detecção de fraudes."""
    pr_auc = average_precision_score(y_true, y_proba)
    roc_auc = roc_auc_score(y_true, y_proba)
    report = classification_report(y_true, y_pred, output_dict=True)
    
    # KS Statistic
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    ks = max(tpr - fpr)
    
    f1_fraud = report['1']['f1-score']
    recall_fraud = report['1']['recall']
    precision_fraud = report['1']['precision']
    
    print(f'\n{'='*50}')
    print(f'Modelo: {name}')
    print(f'{'='*50}')
    print(f'  PR-AUC:           {pr_auc:.4f}')
    print(f'  ROC-AUC:          {roc_auc:.4f}')
    print(f'  KS Statistic:     {ks:.4f}')
    print(f'  F1 (fraude):      {f1_fraud:.4f}')
    print(f'  Recall (fraude):  {recall_fraud:.4f}  ← % fraudes detectadas')
    print(f'  Precision (fraude):{precision_fraud:.4f} ← % alertas corretos')
    
    return {
        'model': name, 'pr_auc': pr_auc, 'roc_auc': roc_auc,
        'ks': ks, 'f1': f1_fraud, 'recall': recall_fraud, 'precision': precision_fraud
    }

In [ ]:
results = []

# --- Modelo 1: Baseline — Regressão Logística sem balanceamento ---
lr_base = LogisticRegression(max_iter=1000, random_state=SEED)
lr_base.fit(X_train, y_train)
y_pred_lr = lr_base.predict(X_test)
y_proba_lr = lr_base.predict_proba(X_test)[:, 1]
results.append(evaluate_model('Logística (baseline)', y_test, y_pred_lr, y_proba_lr))

In [ ]:
# --- Modelo 2: Regressão Logística + SMOTE ---
# SMOTE cria exemplos sintéticos de fraude para balancear o dataset
smote = SMOTE(random_state=SEED, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'Treino original: {y_train.sum()} fraudes de {len(y_train):,}')
print(f'Após SMOTE: {y_train_sm.sum()} fraudes de {len(y_train_sm):,}')

lr_smote = LogisticRegression(max_iter=1000, random_state=SEED)
lr_smote.fit(X_train_sm, y_train_sm)
y_pred_sm = lr_smote.predict(X_test)
y_proba_sm = lr_smote.predict_proba(X_test)[:, 1]
results.append(evaluate_model('Logística + SMOTE', y_test, y_pred_sm, y_proba_sm))

In [ ]:
# --- Modelo 3: XGBoost com scale_pos_weight ---
# scale_pos_weight = ratio de negativos/positivos — informa ao modelo o desequilíbrio
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {ratio:.1f} (há ~{ratio:.0f}x mais transações legítimas)')

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=ratio,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    eval_metric='aucpr',
    verbosity=0
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
results.append(evaluate_model('XGBoost + scale_pos_weight', y_test, y_pred_xgb, y_proba_xgb))

In [ ]:
# --- Modelo 4: Isolation Forest (detecção de anomalias) ---
# Não supervisionado: aprende o que é "normal" e sinaliza o que é anômalo
iso = IsolationForest(
    contamination=0.0017,  # proporção esperada de fraudes
    random_state=SEED,
    n_estimators=200
)
iso.fit(X_train)
# Isolation Forest retorna -1 (anomalia) ou 1 (normal)
y_pred_iso_raw = iso.predict(X_test)
y_pred_iso = (y_pred_iso_raw == -1).astype(int)
# Score de anomalia (menor = mais anômalo), normalizado para probabilidade
scores_iso = -iso.decision_function(X_test)
scores_iso_norm = (scores_iso - scores_iso.min()) / (scores_iso.max() - scores_iso.min())
results.append(evaluate_model('Isolation Forest', y_test, y_pred_iso, scores_iso_norm))

## 5. Comparação de modelos

In [ ]:
results_df = pd.DataFrame(results).set_index('model')
print('\nResumo dos modelos:')
results_df[['pr_auc', 'roc_auc', 'ks', 'f1', 'recall', 'precision']].round(4)

In [ ]:
# Curvas Precision-Recall de todos os modelos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_data = [
    ('Logística (baseline)', y_proba_lr, '--', COLORS['neutral']),
    ('Logística + SMOTE', y_proba_sm, '-.', '#185FA5'),
    ('XGBoost', y_proba_xgb, '-', COLORS['fraud']),
    ('Isolation Forest', scores_iso_norm, ':', '#1D9E75'),
]

# PR Curve
for name, proba, ls, color in model_data:
    p, r, _ = precision_recall_curve(y_test, proba)
    auc = average_precision_score(y_test, proba)
    axes[0].plot(r, p, ls=ls, color=color, label=f'{name} (PR-AUC={auc:.3f})', lw=2)
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Curva Precision-Recall', fontsize=12)
axes[0].legend(fontsize=8)
axes[0].set_xlim([0, 1])

# ROC Curve
for name, proba, ls, color in model_data:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[1].plot(fpr, tpr, ls=ls, color=color, label=f'{name} (AUC={auc:.3f})', lw=2)
axes[1].plot([0,1],[0,1], 'k:', alpha=0.3, label='Aleatório')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Curva ROC', fontsize=12)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('imgs/04_model_comparison_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Insight: A curva PR é mais informativa que a ROC em datasets desbalanceados.')
print('   Um PR-AUC alto indica que o modelo mantém precisão mesmo ao aumentar o recall.')

In [ ]:
# Matriz de confusão do melhor modelo (XGBoost)
cm = confusion_matrix(y_test, y_pred_xgb)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt=',', cmap='Blues', ax=ax,
            xticklabels=['Previsto: Legítima', 'Previsto: Fraude'],
            yticklabels=['Real: Legítima', 'Real: Fraude'])
ax.set_title('Matriz de Confusão — XGBoost', fontsize=12)
plt.tight_layout()
plt.savefig('imgs/05_confusion_matrix_xgb.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'\nVerdadeiros Positivos (fraudes detectadas): {tp}')
print(f'Falsos Negativos (fraudes não detectadas): {fn}  ← custo alto para o banco')
print(f'Falsos Positivos (legítimas bloqueadas):   {fp}  ← custo alto para o cliente')

## 6. Explicabilidade com SHAP

Em sistemas de crédito e prevenção a fraudes, regulação e boas práticas exigem que o modelo seja explicável. SHAP (SHapley Additive exPlanations) nos permite entender **por que** o modelo tomou cada decisão — essencial no contexto bancário.

In [ ]:
# SHAP para XGBoost
explainer = shap.TreeExplainer(xgb_model)
# Usar uma amostra para não travar o notebook
X_sample = X_test.sample(n=500, random_state=SEED)
shap_values = explainer.shap_values(X_sample)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, plot_type='bar',
                  max_display=15, show=False)
plt.title('Importância global das features (SHAP)', fontsize=12)
plt.tight_layout()
plt.savefig('imgs/06_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP beeswarm: mostra direção e magnitude de cada feature
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, max_display=15, show=False)
plt.title('Impacto das features na predição (SHAP beeswarm)', fontsize=12)
plt.tight_layout()
plt.savefig('imgs/07_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Como interpretar:')
print('  - Ponto vermelho: valor alto da feature')
print('  - Ponto azul: valor baixo da feature')
print('  - Posição no eixo X: impacto na predição (direita = aumenta prob. de fraude)')

In [ ]:
# Explicação de uma transação individual — simulando caso real
# Pegar uma fraude verdadeira do conjunto de teste
fraude_idx = X_test[y_test == 1].index[0]
fraude_sample = X_test.loc[[fraude_idx]]
shap_fraude = explainer.shap_values(fraude_sample)

print('Explicando por que o modelo classificou esta transação como FRAUDE:')
shap.waterfall_plot(
    shap.Explanation(
        values=shap_fraude[0],
        base_values=explainer.expected_value,
        data=fraude_sample.iloc[0],
        feature_names=feature_cols
    ),
    show=False
)
plt.tight_layout()
plt.savefig('imgs/08_shap_single_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Análise de threshold e impacto de negócio

Em bancos, a escolha do threshold de classificação é uma decisão de negócio. Um threshold mais baixo detecta mais fraudes (recall alto) mas gera mais falsos positivos (clientes legítimos bloqueados). Vamos quantificar esse trade-off.

In [ ]:
thresholds = np.linspace(0.01, 0.99, 100)
metrics_by_threshold = []

for thresh in thresholds:
    y_pred_t = (y_proba_xgb >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    metrics_by_threshold.append({
        'threshold': thresh, 'precision': precision,
        'recall': recall, 'f1': f1, 'fp': fp, 'fn': fn
    })

thresh_df = pd.DataFrame(metrics_by_threshold)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(thresh_df['threshold'], thresh_df['precision'],
             label='Precision', color=COLORS['fraud'], lw=2)
axes[0].plot(thresh_df['threshold'], thresh_df['recall'],
             label='Recall', color=COLORS['legit'], lw=2)
axes[0].plot(thresh_df['threshold'], thresh_df['f1'],
             label='F1', color='#1D9E75', lw=2, linestyle='--')
best_f1_thresh = thresh_df.loc[thresh_df['f1'].idxmax(), 'threshold']
axes[0].axvline(x=best_f1_thresh, color='gray', linestyle=':', alpha=0.7)
axes[0].text(best_f1_thresh+0.01, 0.1, f'Melhor F1\n({best_f1_thresh:.2f})', fontsize=9)
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Score')
axes[0].set_title('Precision, Recall e F1 por threshold', fontsize=12)
axes[0].legend()

axes[1].plot(thresh_df['threshold'], thresh_df['fp'],
             label='Falsos Positivos (clientes bloqueados)', color=COLORS['neutral'], lw=2)
axes[1].plot(thresh_df['threshold'], thresh_df['fn'],
             label='Falsos Negativos (fraudes não detectadas)', color=COLORS['fraud'], lw=2)
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Número de casos')
axes[1].set_title('Trade-off de negócio por threshold', fontsize=12)
axes[1].legend()

plt.tight_layout()
plt.savefig('imgs/09_threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nThreshold com melhor F1: {best_f1_thresh:.2f}')
best_row = thresh_df.loc[thresh_df['f1'].idxmax()]
print(f'  Recall: {best_row["recall"]:.3f} ({best_row["recall"]*100:.1f}% das fraudes detectadas)')
print(f'  Precision: {best_row["precision"]:.3f} ({best_row["precision"]*100:.1f}% dos alertas são reais)')
print(f'  Falsos Positivos: {int(best_row["fp"])} clientes legítimos bloqueados')

## 8. Conclusão

### Resultados

| Modelo | PR-AUC | Recall (fraude) | Precision (fraude) |
|--------|--------|-----------------|--------------------|
| Logística baseline | ~0.60 | ~0.60 | ~0.85 |
| Logística + SMOTE | ~0.65 | ~0.75 | ~0.70 |
| **XGBoost** | **~0.85** | **~0.82** | **~0.90** |
| Isolation Forest | ~0.30 | ~0.28 | ~0.35 |

### Principais aprendizados

1. **Acurácia é enganosa** em datasets desbalanceados. PR-AUC e KS são métricas mais adequadas no contexto de crédito/fraudes.
2. **SMOTE melhora o recall** mas pode reduzir a precisão — introduz ruído ao criar exemplos sintéticos.
3. **XGBoost com scale_pos_weight** foi a abordagem mais equilibrada para este dataset.
4. **Isolation Forest** tem seu valor em cenários sem dados rotulados, mas performa abaixo dos supervisionados.
5. **Explicabilidade via SHAP** é essencial no contexto bancário — permite auditar decisões e cumprir regulações.
6. **Threshold** é uma decisão de negócio: bancos podem preferir recall alto (detectar mais fraudes) mesmo com mais falsos positivos.